# Records NBA 2026-27: predicciones locas con machine learning

Notebook editorial para seleccionar las 10 historias con más potencial de artículo y guardar una segunda tanda para una continuación. La idea no es predecir el futuro como una casa de apuestas, sino ordenar el radar de récords por una mezcla de rareza estadística, cercanía al récord y factibilidad.

## 1. Pregunta editorial

¿Qué récords de franquicia, positivos o negativos, tienen mejor mezcla de impacto narrativo y posibilidad real de romperse durante la temporada 2026-27?

El modelo produce un `indice_locura` de 0 a 100. Cuanto más alto, más interesante para abrir una historia.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', 140)

PROJECT_DIR = Path(r'C:\Users\rober\OneDrive\Documentos\proyectos_datos\_repos_nebula\records_NBA')
DATA_PATH = PROJECT_DIR / 'radar_records_nba_2026_27.csv'
TOP10_PATH = PROJECT_DIR / 'top10_predicciones_locas_nba.csv'
PART2_PATH = PROJECT_DIR / 'guardadas_segunda_parte_records_nba.csv'
df = pd.read_csv(DATA_PATH)
df.head()

## 2. Comprobación rápida del radar

El dataset parte de 90 pistas: 30 franquicias, positivos y negativos, con jugadores y equipos mezclados.

In [ ]:
resumen = pd.DataFrame({
    'filas': [len(df)],
    'equipos': [df['equipo'].nunique()],
    'positivos': [(df['polaridad'] == 'Positivo').sum()],
    'negativos': [(df['polaridad'] == 'Negativo').sum()],
    'jugador': [(df['nivel'] == 'Jugador').sum()],
    'equipo': [(df['nivel'] == 'Equipo').sum()],
})
resumen

## 3. Limpieza y variables para el modelo

La gracia está en combinar tres señales: qué tan raro es el caso, qué tan cerca está del umbral y qué tan viable parece según el radar inicial.

In [ ]:
numeric_cols = [
    'baseline_2025_26', 'partidos_2025_26', 'record_vigente', 'objetivo_para_romper',
    'distancia_vs_2025_26', 'ritmo_objetivo_por_partido', 'indice_factibilidad'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['distancia_abs'] = df['distancia_vs_2025_26'].abs()
df['ratio_objetivo_record'] = df['objetivo_para_romper'] / df['record_vigente'].replace(0, np.nan)
df['ratio_baseline_objetivo'] = df['baseline_2025_26'] / df['objetivo_para_romper'].replace(0, np.nan)
df['cerca_del_record'] = 1 - (df['distancia_abs'] / df['objetivo_para_romper'].replace(0, np.nan))
df['cerca_del_record'] = df['cerca_del_record'].clip(lower=0, upper=1)
df['negativo_flag'] = (df['polaridad'] == 'Negativo').astype(int)
df['jugador_flag'] = (df['nivel'] == 'Jugador').astype(int)
df[['equipo', 'protagonista', 'polaridad', 'nivel', 'metrica', 'distancia_abs', 'cerca_del_record']].head()

## 4. Modelo ML

Uso un clustering K-Means autocontenido. La distancia de cada caso al centroide de su grupo mide rareza; luego se combina con factibilidad, cercanía al récord y polaridad para crear `indice_locura`.

In [ ]:
def minmax(series):
    values = pd.to_numeric(series, errors='coerce').astype(float)
    lo, hi = values.min(), values.max()
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series(np.zeros(len(values)), index=series.index)
    return (values - lo) / (hi - lo)

def make_features(data, numeric_features, categorical_features):
    numeric = data[numeric_features].copy()
    for col in numeric.columns:
        numeric[col] = pd.to_numeric(numeric[col], errors='coerce')
        numeric[col] = numeric[col].fillna(numeric[col].median())
        numeric[col] = minmax(numeric[col])
    categorical = pd.get_dummies(data[categorical_features].fillna('Sin dato'), dtype=float)
    return pd.concat([numeric, categorical], axis=1).to_numpy(dtype=float)

def kmeans_numpy(X, k=6, iterations=80, seed=42):
    rng = np.random.default_rng(seed)
    centroids = X[rng.choice(len(X), size=k, replace=False)].copy()
    labels = np.zeros(len(X), dtype=int)
    for _ in range(iterations):
        distances = ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
        labels = distances.argmin(axis=1)
        new_centroids = centroids.copy()
        for cluster in range(k):
            if np.any(labels == cluster):
                new_centroids[cluster] = X[labels == cluster].mean(axis=0)
        if np.allclose(centroids, new_centroids):
            break
        centroids = new_centroids
    distances = np.sqrt(((X - centroids[labels]) ** 2).sum(axis=1))
    return labels, distances

feature_cols = [
    'baseline_2025_26', 'partidos_2025_26', 'record_vigente', 'objetivo_para_romper',
    'distancia_abs', 'ritmo_objetivo_por_partido', 'indice_factibilidad',
    'ratio_objetivo_record', 'ratio_baseline_objetivo', 'cerca_del_record',
    'negativo_flag', 'jugador_flag'
]
categorical_features = ['equipo', 'metrica_codigo', 'nivel', 'polaridad', 'factibilidad']

X = make_features(df, feature_cols, categorical_features)
labels, distances = kmeans_numpy(X, k=6)
df['cluster_ml'] = labels
df['rareza_ml'] = minmax(pd.Series(distances, index=df.index))

df['score_ml_predicho'] = (
    0.48 * df['rareza_ml']
    + 0.34 * df['indice_factibilidad'].fillna(df['indice_factibilidad'].median())
    + 0.12 * df['cerca_del_record'].fillna(0)
    + 0.06 * df['negativo_flag']
)
df['score_ml_predicho'] = minmax(df['score_ml_predicho'])

df['indice_locura'] = (
    0.50 * df['score_ml_predicho']
    + 0.30 * df['rareza_ml']
    + 0.20 * df['indice_factibilidad'].fillna(df['indice_factibilidad'].median())
)
df['indice_locura'] = (100 * minmax(df['indice_locura'])).round(1)
df[['equipo', 'protagonista', 'polaridad', 'cluster_ml', 'indice_locura', 'rareza_ml']].sort_values('indice_locura', ascending=False).head(10)

## 5. Ranking editorial

El ranking prioriza historias con buen ángulo de titular. Después se puede filtrar manualmente para equilibrar estrellas, franquicias, récords positivos y desastres históricos.

In [ ]:
df['lectura_ml'] = df.apply(
    lambda row: (
        f"{row['protagonista']} combina rareza ML {row['rareza_ml']:.2f}, "
        f"factibilidad {row['indice_factibilidad']:.2f} y una distancia de "
        f"{row['distancia_vs_2025_26']:.0f} respecto al umbral."
    ),
    axis=1,
)

editorial_cols = [
    'equipo', 'abreviatura', 'polaridad', 'nivel', 'protagonista', 'metrica',
    'baseline_2025_26', 'record_vigente', 'objetivo_para_romper', 'distancia_vs_2025_26',
    'ritmo_objetivo_por_partido', 'factibilidad', 'indice_factibilidad',
    'rareza_ml', 'score_ml_predicho', 'indice_locura', 'gancho_editorial', 'lectura_ml',
    'source_record_url', 'source_baseline_url'
]
ranked = df.sort_values(['indice_locura', 'indice_factibilidad', 'rareza_ml'], ascending=[False, False, False]).reset_index(drop=True)
ranked.insert(0, 'ranking_ml', ranked.index + 1)
top10 = ranked[['ranking_ml'] + editorial_cols].head(10)
segunda_parte = ranked[['ranking_ml'] + editorial_cols].iloc[10:25]
top10

## 6. Las 10 predicciones más locas

| ranking_ml | equipo | polaridad | nivel | protagonista | metrica | indice_locura | gancho_editorial |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | Chicago Bulls | Positivo | Jugador | Josh Giddey | triple-dobles | 100.0 | Josh Giddey necesita 16 triple-dobles; en 2025-26 firmó 13 en 54 partidos. |
| 2 | Sacramento Kings | Positivo | Equipo | Sacramento Kings | puntos | 90.8 | Sacramento Kings viene de 9102 puntos; el récord es 9898 (2022-23) y exige 9899. |
| 3 | Portland Trail Blazers | Positivo | Jugador | Donovan Clingan | rebotes | 89.1 | Donovan Clingan necesita 968 rebotes; en 2025-26 firmó 892 en 77 partidos. |
| 4 | Oklahoma City Thunder | Positivo | Jugador | Shai Gilgeous-Alexander | puntos | 81.2 | Shai Gilgeous-Alexander necesita 2594 puntos; en 2025-26 firmó 2117 en 68 partidos. |
| 5 | New York Knicks | Positivo | Jugador | Jalen Brunson | puntos | 79.7 | Jalen Brunson necesita 2348 puntos; en 2025-26 firmó 1927 en 74 partidos. |
| 6 | Minnesota Timberwolves | Positivo | Jugador | Anthony Edwards | puntos | 79.7 | Anthony Edwards necesita 2178 puntos; en 2025-26 firmó 1757 en 61 partidos. |
| 7 | Houston Rockets | Positivo | Jugador | Kevin Durant | puntos | 75.5 | Kevin Durant necesita 2819 puntos; en 2025-26 firmó 2026 en 78 partidos. |
| 8 | Cleveland Cavaliers | Positivo | Equipo | Cleveland Cavaliers | asistencias | 75.4 | Cleveland Cavaliers viene de 2321 asistencias; el récord es 2349 (1992-93) y exige 2350. |
| 9 | San Antonio Spurs | Positivo | Jugador | Julian Champagnie | triples anotados | 75.3 | Julian Champagnie ya posee el récord de triples anotados con 195; con 196 lo mejora. |
| 10 | Charlotte Hornets | Positivo | Jugador | Kon Knueppel | triples anotados | 75.2 | Kon Knueppel ya posee el récord de triples anotados con 273; con 274 lo mejora. |

In [ ]:
grafico = top10[['ranking_ml', 'protagonista', 'abreviatura', 'polaridad', 'indice_locura']].copy()
grafico['barra'] = grafico['indice_locura'].apply(lambda x: '█' * int(round(x / 5)))
grafico[['ranking_ml', 'protagonista', 'abreviatura', 'polaridad', 'indice_locura', 'barra']]

## 7. Banco para segunda parte

Estas quedan guardadas como segunda tanda. Funcionan bien para una continuación, una pieza tipo ranking extendido o recuadros laterales.

In [ ]:
segunda_parte

## 8. Exportación

Se guardan dos CSV auxiliares junto al notebook para que puedas escribir la pieza sin volver a ejecutar todo.

In [ ]:
top10.to_csv(TOP10_PATH, index=False, encoding='utf-8-sig')
segunda_parte.to_csv(PART2_PATH, index=False, encoding='utf-8-sig')
print(f'Top 10 guardado en: {TOP10_PATH}')
print(f'Segunda parte guardada en: {PART2_PATH}')

## 9. Nota de uso

El modelo ordena el radar, pero la selección final debe pasar por criterio editorial: variedad de equipos, nombres reconocibles, equilibrio entre positivo y negativo y solidez de fuente. Antes de publicar, conviene auditar de nuevo los récords históricos más antiguos o de franquicias trasladadas.